# Sistema experto con interfaz web: diagnóstico de rendimiento en videojuegos de PC

**Inteligencia Artificial II — Proyecto final**
Fundación Universitaria Los Libertadores

**Autores:** Juan F. Vanegas Silva · Diego Felipe Almanza Vanegas
**Fecha:** noviembre de 2026

Este notebook documenta y valida el sistema completo. **No duplica el código**: importa los mismos módulos de `src/` que usa la aplicación web, así que lo que se prueba aquí es exactamente lo que corre en Flask.

| Archivo | Qué contiene |
|---|---|
| `src/base_conocimiento.py` | 11 predicados observables, 14 derivados y 15 reglas con FC |
| `src/entrada.py` | Validación del formulario y conversión de lecturas numéricas a hechos (umbrales) |
| `src/motor_inferencia.py` | Encadenamiento hacia adelante, combinación de FC, explicación (`porque`, `porque_no`), detección de ciclos, logging |
| `src/casos.py` | Los 9 casos de prueba (los mismos que se precargan en la interfaz) |
| `src/app.py` + `templates/` + `static/` | Aplicación web Flask (backend + frontend) |

> **Requisitos:** Python 3.8+ y `pip install -r requirements.txt` (solo Flask; el motor usa únicamente la biblioteca estándar). Ejecutar el notebook desde la carpeta del proyecto.

## 1. Introducción

### Descripción del dominio
Cuando un videojuego "va mal" en el PC el jugador solo ve el síntoma (pocos FPS, tirones o lag), pero la causa puede ser muy distinta: la GPU al límite, el procesador saturado, sobrecalentamiento, falta de RAM, el disco o la red. Cada causa se arregla de forma diferente: comprar otra GPU no arregla un problema de temperatura y bajar gráficos no arregla el lag del WiFi.

El sistema captura cómo razona un jugador con experiencia en hardware: lee las lecturas de monitoreo (FPS, % de GPU/CPU/RAM, temperatura, ping) y los síntomas, identifica la causa probable y recomienda la acción concreta, explicando por qué.

### Justificación
- Es un problema real y frecuente; las lecturas se obtienen gratis con MSI Afterburner o el administrador de tareas.
- El conocimiento se expresa naturalmente como reglas SI-ENTONCES y el diagnóstico es incierto (un síntoma puede tener varias causas), lo que justifica los factores de certeza.

### Qué cambió frente a la Actividad 4 (SBC en notebook)
| Actividad 4 | Proyecto final |
|---|---|
| 12 reglas | **15 reglas** (R13 combinación de evidencia, R14 negación, R15 recomendación de servidor) |
| El usuario decidía si un valor era "alto" | **Lecturas numéricas** convertidas a hechos con umbrales (`entrada.py`) |
| Hechos verdadero/ausente | **Tres estados**: verdadero, falso (afirmado) y desconocido |
| No combinaba evidencias | **Combinación MYCIN**: $FC = FC_1 + FC_2(1-FC_1)$ |
| Sin negación | Reglas con $\neg P(x)$ (R14: ping alto **sin** WiFi → servidor) |
| Solo `porque()` | `porque()` + **`porque_no()`** (qué condición faltó) |
| Sin protección de ciclos | Refracción + tope de iteraciones + `detectar_ciclos()` |
| Notebook | **Aplicación web Flask** con explicación interactiva y casos precargados |

### Alcance
| Cubre | No cubre |
|---|---|
| Cuello de botella de GPU, CPU y disco | Fallas físicas (pantallazos azules, artefactos) |
| Sobrecalentamiento (*thermal throttling*) | Drivers, sistema operativo y bugs del juego |
| Falta de memoria RAM | Fuente de poder, resolución, consolas y celulares |
| Lag por WiFi y lag del servidor/proveedor | Diagnóstico del router o del proveedor en detalle |

## 2. Diseño del sistema

```
 ┌───────────────────── Navegador (frontend) ─────────────────────┐
 │ templates/index.html + static/style.css + static/script.js     │
 │ Formulario ─► validación JS ─► fetch POST /api/diagnosticar    │
 │ Tarjetas de diagnóstico · árbol "¿Por qué?" · "¿Por qué no?"   │
 └──────────────────────────────┬─────────────────────────────────┘
                                │ JSON
 ┌──────────────────────────────▼──── Flask (src/app.py) ──────────┐
 │ /api/base   /api/diagnosticar   /api/porque                     │
 │        │                                                         │
 │  entrada.py ──► hechos iniciales {predicado: True/False}         │
 │        │                                                         │
 │  motor_inferencia.py ◄── base_conocimiento.py (15 reglas)        │
 │   match → disparo (Modus Ponens + FC) → punto fijo               │
 │   explicar_razonamiento · porque · porque_no · detectar_ciclos   │
 └─────────────────────────────────────────────────────────────────┘
```

**Flujo de datos:** formulario → JSON → validación y umbrales (`entrada.py`) → hechos iniciales (FC = 1.0) → motor (memoria de trabajo + agenda) → hechos finales + reglas aplicadas → explicación → JSON → interfaz.

**Decisión clave:** la interfaz no contiene lógica del SBC; solo envía datos y dibuja la respuesta. Por eso el notebook, las pruebas y la web dan exactamente los mismos resultados.

In [1]:
import sys, logging
from pathlib import Path
from IPython.display import Markdown, display

sys.path.insert(0, str(Path("src").resolve()))
logging.basicConfig(level=logging.WARNING)   # el log del motor se muestra desde el resultado, no por consola

from base_conocimiento import crear_base_conocimiento, crear_regla, si_entonces, PREDICADOS, CONCLUSIONES, REGLAS
from entrada import hechos_desde_formulario, ErrorEntrada, CAMPOS_NUMERICOS
from motor_inferencia import (encadenamiento_adelante, ejecutar_inferencia, explicar_razonamiento,
                              porque, porque_no, detectar_ciclos, diagnosticar, combinar_certezas)
from casos import CASOS

BASE = crear_base_conocimiento()
print(f"Predicados observables: {len(PREDICADOS)} | derivados: {len(CONCLUSIONES)} | reglas: {len(REGLAS)}")

Predicados observables: 11 | derivados: 14 | reglas: 15


## 3. Base de conocimiento

### 3.1 Predicados del dominio
Todos los predicados son de **aridad 1**: $P: E \rightarrow \{V, F\}$, donde $E$ es el conjunto de equipos (PC) en una sesión de juego y $x \in E$.
Los 11 **predicados observables** son los hechos de entrada; cada uno se calcula a partir de un campo del formulario:

In [2]:
filas = ["| Predicado | Tipo | Dominio → Rango | Significado | Se calcula con |", "|---|---|---|---|---|"]
for p, d in PREDICADOS.items():
    filas.append(f"| `{p}(x)` | observable | E → {{V, F}} | {d['descripcion']} | `{d['umbral']}` |")
display(Markdown("\n".join(filas)))

| Predicado | Tipo | Dominio → Rango | Significado | Se calcula con |
|---|---|---|---|---|
| `FPSBajos(x)` | observable | E → {V, F} | El juego corre a menos de 30 FPS | `fps < 30` |
| `UsoGPUAlto(x)` | observable | E → {V, F} | La GPU está al 95 % de uso o más mientras se juega | `uso_gpu ≥ 95 %` |
| `UsoCPUAlto(x)` | observable | E → {V, F} | La CPU está al 90 % de uso o más mientras se juega | `uso_cpu ≥ 90 %` |
| `GraficosAltos(x)` | observable | E → {V, F} | El juego está configurado en calidad alta o ultra | `calidad ∈ {alta, ultra}` |
| `TempAlta(x)` | observable | E → {V, F} | La CPU o la GPU supera los 90 °C | `temperatura > 90 °C` |
| `FPSCaenConTiempo(x)` | observable | E → {V, F} | Los FPS empiezan bien y bajan después de varios minutos de juego | `respuesta = sí` |
| `Tirones(x)` | observable | E → {V, F} | Hay congelamientos breves (stuttering) durante el juego | `respuesta = sí` |
| `RAMLlena(x)` | observable | E → {V, F} | El uso de memoria RAM es del 90 % o más | `uso_ram ≥ 90 %` |
| `JuegoEnHDD(x)` | observable | E → {V, F} | El juego está instalado en un disco duro mecánico (no SSD) | `almacenamiento = HDD` |
| `PingAlto(x)` | observable | E → {V, F} | La latencia en juegos en línea supera los 100 ms | `ping > 100 ms` |
| `ConexionWiFi(x)` | observable | E → {V, F} | El equipo se conecta a internet por WiFi | `conexión = WiFi` |

Los **predicados derivados** los concluye el motor (mismo dominio y rango). Se dividen en diagnósticos y recomendaciones:

In [3]:
filas = ["| Predicado | Tipo | Texto para el usuario |", "|---|---|---|"]
for p, d in CONCLUSIONES.items():
    filas.append(f"| `{p}(x)` | {d['tipo']} | {d['titulo']} |")
display(Markdown("\n".join(filas)))

| Predicado | Tipo | Texto para el usuario |
|---|---|---|
| `CuelloBotellaGPU(x)` | diagnostico | La tarjeta gráfica (GPU) limita el rendimiento |
| `CuelloBotellaCPU(x)` | diagnostico | El procesador (CPU) limita el rendimiento |
| `ThermalThrottling(x)` | diagnostico | Sobrecalentamiento (thermal throttling) |
| `FaltaRAM(x)` | diagnostico | La memoria RAM no es suficiente |
| `CuelloBotellaDisco(x)` | diagnostico | El disco duro no carga los datos a tiempo |
| `ProblemaRedLocal(x)` | diagnostico | El lag viene de la conexión WiFi |
| `ProblemaServidor(x)` | diagnostico | El lag viene del servidor o del proveedor de internet |
| `RecBajarGraficos(x)` | recomendacion | Bajar la calidad gráfica o activar DLSS/FSR |
| `RecCerrarProcesos(x)` | recomendacion | Cerrar programas en segundo plano |
| `RecMantenimientoTermico(x)` | recomendacion | Hacer mantenimiento térmico |
| `RecAmpliarRAM(x)` | recomendacion | Ampliar la memoria RAM |
| `RecMoverASSD(x)` | recomendacion | Mover el juego a un SSD |
| `RecCableEthernet(x)` | recomendacion | Conectar el PC por cable Ethernet |
| `RecCambiarServidor(x)` | recomendacion | Elegir un servidor más cercano o revisar el proveedor |

### 3.2 Reglas en lógica de predicados y en formato SI-ENTONCES
Todas las reglas están **cuantificadas universalmente** ($\forall x$): valen para cualquier equipo. Para pasarlas a reglas de producción se elimina el $\forall x$ y el motor instancia $x$ con el equipo consultado (**instanciación universal**); luego aplica **Modus Ponens**: si las condiciones son verdaderas, la conclusión es verdadera.

El $\exists$ aparece en las **consultas**: `porque()` responde preguntas como $\exists x\, ThermalThrottling(x)$ — "¿hay algún equipo con sobrecalentamiento y por qué?".

In [4]:
filas = ["| Regla | Grupo | Lógica de predicados | SI-ENTONCES | FC | Justificación |", "|---|---|---|---|---|---|"]
for r in REGLAS:
    filas.append(f"| {r['nombre']} | {r['categoria']} | {r['logica']} | `{si_entonces(r)}` | {r['certeza']:.2f} | {r['justificacion']} |")
display(Markdown("\n".join(filas)))

| Regla | Grupo | Lógica de predicados | SI-ENTONCES | FC | Justificación |
|---|---|---|---|---|---|
| R1 | Hardware | ∀x (FPSBajos(x) ∧ UsoGPUAlto(x) → CuelloBotellaGPU(x)) | `SI FPSBajos Y UsoGPUAlto ENTONCES CuelloBotellaGPU` | 0.85 | Si la GPU ya está al máximo y no alcanza 30 FPS, ella es el límite. |
| R2 | Hardware | ∀x (FPSBajos(x) ∧ UsoCPUAlto(x) → CuelloBotellaCPU(x)) | `SI FPSBajos Y UsoCPUAlto ENTONCES CuelloBotellaCPU` | 0.80 | Con el procesador saturado, la GPU espera los datos de cada cuadro. |
| R3 | Hardware | ∀x (CuelloBotellaGPU(x) ∧ GraficosAltos(x) → RecBajarGraficos(x)) | `SI CuelloBotellaGPU Y GraficosAltos ENTONCES RecBajarGraficos` | 0.90 | Bajar la calidad gráfica le quita carga directamente a la GPU. |
| R4 | Hardware | ∀x (CuelloBotellaCPU(x) → RecCerrarProcesos(x)) | `SI CuelloBotellaCPU ENTONCES RecCerrarProcesos` | 0.70 | Los programas en segundo plano le roban tiempo de CPU al juego. |
| R5 | Temperatura | ∀x (TempAlta(x) ∧ FPSCaenConTiempo(x) → ThermalThrottling(x)) | `SI TempAlta Y FPSCaenConTiempo ENTONCES ThermalThrottling` | 0.85 | El calor acumulado obliga al hardware a bajar su frecuencia. |
| R6 | Temperatura | ∀x (ThermalThrottling(x) → RecMantenimientoTermico(x)) | `SI ThermalThrottling ENTONCES RecMantenimientoTermico` | 0.90 | Limpiar ventiladores y cambiar la pasta térmica baja la temperatura. |
| R7 | Memoria y disco | ∀x (Tirones(x) ∧ RAMLlena(x) → FaltaRAM(x)) | `SI Tirones Y RAMLlena ENTONCES FaltaRAM` | 0.80 | Sin RAM libre el sistema usa el archivo de paginación y se congela. |
| R8 | Memoria y disco | ∀x (FaltaRAM(x) → RecAmpliarRAM(x)) | `SI FaltaRAM ENTONCES RecAmpliarRAM` | 0.75 | Más memoria evita el uso del disco como memoria virtual. |
| R9 | Memoria y disco | ∀x (Tirones(x) ∧ JuegoEnHDD(x) → CuelloBotellaDisco(x)) | `SI Tirones Y JuegoEnHDD ENTONCES CuelloBotellaDisco` | 0.70 | Un HDD es lento para cargar datos sobre la marcha (FC bajo: los tirones también pueden venir de la compilación de shaders). |
| R10 | Memoria y disco | ∀x (CuelloBotellaDisco(x) → RecMoverASSD(x)) | `SI CuelloBotellaDisco ENTONCES RecMoverASSD` | 0.85 | Un SSD carga los datos decenas de veces más rápido que un HDD. |
| R11 | Red | ∀x (PingAlto(x) ∧ ConexionWiFi(x) → ProblemaRedLocal(x)) | `SI PingAlto Y ConexionWiFi ENTONCES ProblemaRedLocal` | 0.75 | El WiFi agrega latencia e interferencia frente al cable. |
| R12 | Red | ∀x (ProblemaRedLocal(x) → RecCableEthernet(x)) | `SI ProblemaRedLocal ENTONCES RecCableEthernet` | 0.85 | El cable da una conexión estable y con menos latencia. |
| R13 | Memoria y disco | ∀x (FaltaRAM(x) → RecCerrarProcesos(x)) | `SI FaltaRAM ENTONCES RecCerrarProcesos` | 0.60 | Cerrar programas libera memoria. Comparte conclusión con R4: si ambas se disparan, el motor combina sus certezas (fórmula de MYCIN). |
| R14 | Red | ∀x (PingAlto(x) ∧ ¬ConexionWiFi(x) → ProblemaServidor(x)) | `SI PingAlto Y NO ConexionWiFi ENTONCES ProblemaServidor` | 0.60 | Descartada la red inalámbrica, la latencia se origina fuera del equipo (FC bajo: también podría ser el router). |
| R15 | Red | ∀x (ProblemaServidor(x) → RecCambiarServidor(x)) | `SI ProblemaServidor ENTONCES RecCambiarServidor` | 0.70 | Un servidor más cercano reduce la distancia que recorren los paquetes. |

### 3.3 Factores de certeza
- Hechos observados: $FC = 1.0$ (el usuario los midió).
- Regla aplicada: $FC(\text{conclusión}) = FC(\text{regla}) \times \min(FC(\text{condiciones}))$. Se usa el mínimo porque una conjunción es tan confiable como su condición más débil.
- **Dos reglas, misma conclusión** (nuevo): $FC = FC_1 + FC_2\,(1 - FC_1)$ (MYCIN). Dos evidencias independientes refuerzan la conclusión, pero nunca pasan de 1.
- La certeza **baja** a medida que la cadena se alarga: una recomendación depende de que el diagnóstico sea correcto.

Los FC bajos tienen razón: R9 = 0.70 porque los tirones con HDD también pueden venir de la compilación de *shaders*; R14 = 0.60 porque el ping alto por cable también podría ser el router.

In [5]:
print("Ejemplo de combinación (caso 5): R4 da 0.56 y R13 da 0.48 para RecCerrarProcesos")
print("FC combinado = 0.56 + 0.48·(1 − 0.56) =", combinar_certezas(0.56, 0.48))

Ejemplo de combinación (caso 5): R4 da 0.56 y R13 da 0.48 para RecCerrarProcesos
FC combinado = 0.56 + 0.48·(1 − 0.56) = 0.77


### 3.4 Verificación estática de la base: ¿hay ciclos?
`detectar_ciclos()` construye el grafo *predicado condición → predicado conclusión* y lo recorre en profundidad (DFS con colores blanco/gris/negro). Un arco hacia un nodo "gris" (que está en el camino actual) es un ciclo.

In [6]:
print("Ciclos en la base real:", detectar_ciclos(BASE) or "ninguno")

# Base de juguete con un ciclo, para mostrar que se detecta y que el motor igual termina
ciclica = [crear_regla("A", "t", [("P", True)], "Q", 0.9, "", "", ""),
           crear_regla("B", "t", [("Q", True)], "P", 0.9, "", "", "")]
print("Ciclos en la base de juguete:", detectar_ciclos(ciclica))
res = ejecutar_inferencia(ciclica, {"P": True})
print(f"El motor terminó en {res.iteraciones} iteraciones (punto fijo = {res.punto_fijo}) gracias a la refracción.")

Ciclos en la base real: ninguno
Ciclos en la base de juguete: [['P', 'Q', 'P']]
El motor terminó en 3 iteraciones (punto fijo = True) gracias a la refracción.


## 4. Capa de entrada: de lecturas numéricas a hechos
El usuario escribe la lectura (por ejemplo, GPU = 99 %) y `entrada.py` aplica el umbral del predicado. Lo que se deja vacío o en "No sé" queda **desconocido**: no es verdadero ni falso. Esto importa para la negación: R14 pide $\neg ConexionWiFi(x)$ y solo se dispara si el usuario **afirmó** que está por cable.

In [7]:
for campo, (pred, _, mn, mx, unidad, etiqueta) in CAMPOS_NUMERICOS.items():
    print(f"{etiqueta:<20} [{mn}–{mx} {unidad}] -> {pred:<11} umbral: {PREDICADOS[pred]['umbral']}")

hechos, lecturas = hechos_desde_formulario({"fps": 30, "uso_gpu": 95, "conexion": "cable", "tirones": ""})
print("\nEntrada {fps: 30, uso_gpu: 95, conexion: cable, tirones: 'No sé'} ->", hechos)

for malo in ({}, {"uso_gpu": 150}, {"fps": "rápido"}):
    try:
        hechos_desde_formulario(malo)
    except ErrorEntrada as e:
        print(f"Entrada {malo!s:<18} -> error: {e.errores}")

FPS promedio         [0–1000 FPS] -> FPSBajos    umbral: fps < 30
Uso de GPU           [0–100 %] -> UsoGPUAlto  umbral: uso_gpu ≥ 95 %
Uso de CPU           [0–100 %] -> UsoCPUAlto  umbral: uso_cpu ≥ 90 %
Temperatura máxima   [0–150 °C] -> TempAlta    umbral: temperatura > 90 °C
Uso de RAM           [0–100 %] -> RAMLlena    umbral: uso_ram ≥ 90 %
Ping                 [0–5000 ms] -> PingAlto    umbral: ping > 100 ms

Entrada {fps: 30, uso_gpu: 95, conexion: cable, tirones: 'No sé'} -> {'FPSBajos': False, 'UsoGPUAlto': True, 'ConexionWiFi': False}
Entrada {}                 -> error: ['Ingrese al menos una lectura o síntoma para poder diagnosticar.']
Entrada {'uso_gpu': 150}   -> error: ['Uso de GPU: debe estar entre 0 y 100 %.']
Entrada {'fps': 'rápido'}  -> error: ['FPS promedio: «rápido» no es un número.']


## 5. Motor de inferencia

### 5.1 Algoritmo de encadenamiento hacia adelante
Ciclo de reconocimiento-acción hasta un **punto fijo**:
1. **Match:** se recorren las reglas que aún no se han disparado y se buscan aquellas cuyas condiciones se cumplen en la memoria de trabajo (instanciando $x$).
2. **Resolución de conflictos:** se disparan **todas** las reglas activadas, en el orden de la base, usando los hechos del **inicio** de la iteración. Así cada iteración corresponde a un nivel de la cadena: iteración 1 = diagnósticos, iteración 2 = recomendaciones.
3. **Act (Modus Ponens):** se agrega la conclusión con su FC; si ya existía, se combinan las certezas (MYCIN).
4. **Refracción:** cada regla se dispara una sola vez. Con esto el motor termina como máximo en $|R| + 1$ iteraciones aunque la base tenga ciclos; además hay un tope `MAX_ITERACIONES = 50`.
5. Si en una iteración no se activa nada, el motor se detiene. Si no se aplicó ninguna regla, se informa "sin conclusión".

Núcleo del motor (fuente real de `ejecutar_inferencia`, mostrado con `inspect`):

In [8]:
import inspect, motor_inferencia
fuente = inspect.getsource(motor_inferencia.ejecutar_inferencia)
print(fuente[fuente.index("    iteracion, punto_fijo"):fuente.index("    if not punto_fijo")])

    iteracion, punto_fijo = 0, False
    while iteracion < max_iteraciones:
        iteracion += 1

        # MATCH: reglas activadas con la memoria al inicio de la iteración
        agenda = []
        for regla in reglas:
            if regla["nombre"] in disparadas:
                continue                                   # refracción
            if all(cumple(c, memoria) for c in regla["condiciones"]):
                condiciones = [{"predicado": p, "valor": v, "certeza": memoria[p]["certeza"]}
                               for p, v in regla["condiciones"]]
                fc_condiciones = min(c["certeza"] for c in condiciones) if condiciones else 1.0
                agenda.append((regla, condiciones, round(regla["certeza"] * fc_condiciones, 2)))

        if not agenda:
            punto_fijo = True
            registrar(logging.INFO, f"Iteración {iteracion}: ninguna regla nueva se activa -> punto fijo.")
            break

        registrar(logging.INFO, f"Iteración {iteracion}

### 5.2 Ejemplo completo paso a paso (caso 5: PC viejo)

In [9]:
caso = CASOS[4]
hechos, lecturas = hechos_desde_formulario(caso["datos"])
resultado = ejecutar_inferencia(BASE, hechos, caso["equipo"])
print("Datos del formulario:", caso["datos"])
print("Hechos iniciales:", hechos, "\n")
print("Registro (log) del motor:")
print("\n".join(resultado.log))

Datos del formulario: {'fps': 20, 'uso_cpu': 97, 'tirones': 'si', 'uso_ram': 92, 'almacenamiento': 'hdd'}
Hechos iniciales: {'FPSBajos': True, 'UsoCPUAlto': True, 'RAMLlena': True, 'JuegoEnHDD': True, 'Tirones': True} 

Registro (log) del motor:
Inicio: FPSBajos(pc_viejo), UsoCPUAlto(pc_viejo), RAMLlena(pc_viejo), JuegoEnHDD(pc_viejo), Tirones(pc_viejo)
Iteración 1: reglas activadas R2, R7, R9
  R2 dispara: CuelloBotellaCPU(pc_viejo) [FC=0.8]
  R7 dispara: FaltaRAM(pc_viejo) [FC=0.8]
  R9 dispara: CuelloBotellaDisco(pc_viejo) [FC=0.7]
Iteración 2: reglas activadas R4, R8, R10, R13
  R4 dispara: RecCerrarProcesos(pc_viejo) [FC=0.56]
  R8 dispara: RecAmpliarRAM(pc_viejo) [FC=0.6]
  R10 dispara: RecMoverASSD(pc_viejo) [FC=0.59]
  R13 refuerza RecCerrarProcesos(pc_viejo): 0.56 + 0.48·(1 − 0.56) = 0.77 (combinación MYCIN)
Iteración 3: ninguna regla nueva se activa -> punto fijo.


### 5.3 Mecanismo de explicación
- `explicar_razonamiento(reglas_aplicadas)`: la cadena completa, regla por regla, con el cálculo del FC.
- `porque(conclusion, hechos_iniciales, reglas_aplicadas)`: recorre **hacia atrás** desde la conclusión hasta los hechos observados (árbol de justificación). Si dos reglas soportan la conclusión, muestra ambas ramas.
- `porque_no(conclusion, hechos_finales, base)`: para algo que **no** se concluyó, lista cada regla que lo concluiría y el estado de cada condición (se cumple / es falsa / dato no informado / no se concluyó).

In [10]:
print(explicar_razonamiento(resultado.reglas_aplicadas, caso["equipo"]))

Iteración 1 | R2: FPSBajos(pc_viejo) ∧ UsoCPUAlto(pc_viejo) → CuelloBotellaCPU(pc_viejo)  [FC = 0.8 × min(1.0, 1.0) = 0.8]
Iteración 1 | R7: Tirones(pc_viejo) ∧ RAMLlena(pc_viejo) → FaltaRAM(pc_viejo)  [FC = 0.8 × min(1.0, 1.0) = 0.8]
Iteración 1 | R9: Tirones(pc_viejo) ∧ JuegoEnHDD(pc_viejo) → CuelloBotellaDisco(pc_viejo)  [FC = 0.7 × min(1.0, 1.0) = 0.7]
Iteración 2 | R4: CuelloBotellaCPU(pc_viejo) → RecCerrarProcesos(pc_viejo)  [FC = 0.7 × min(0.8) = 0.56]
Iteración 2 | R8: FaltaRAM(pc_viejo) → RecAmpliarRAM(pc_viejo)  [FC = 0.75 × min(0.8) = 0.6]
Iteración 2 | R10: CuelloBotellaDisco(pc_viejo) → RecMoverASSD(pc_viejo)  [FC = 0.85 × min(0.7) = 0.59]
Iteración 2 | R13: FaltaRAM(pc_viejo) → RecCerrarProcesos(pc_viejo)  [FC = 0.6 × min(0.8) = 0.48]  ⇒ combinada con la evidencia previa: 0.56 + 0.48·(1 − 0.56) = 0.77


In [11]:
print("¿Por qué se recomienda cerrar procesos? (dos reglas lo soportan)\n")
print(porque("RecCerrarProcesos", hechos, resultado.reglas_aplicadas, caso["equipo"]))

¿Por qué se recomienda cerrar procesos? (dos reglas lo soportan)

• RecCerrarProcesos(pc_viejo) [FC = 0.77] se concluyó con R4 y R13, porque se cumplen:
    (R4, FC del paso = 0.56)
    • CuelloBotellaCPU(pc_viejo) [FC = 0.8] se concluyó con R2, porque se cumplen:
        • FPSBajos(pc_viejo) es un hecho observado (FC = 1.0).
        • UsoCPUAlto(pc_viejo) es un hecho observado (FC = 1.0).
    (R13, FC del paso = 0.48)
    • FaltaRAM(pc_viejo) [FC = 0.8] se concluyó con R7, porque se cumplen:
        • Tirones(pc_viejo) es un hecho observado (FC = 1.0).
        • RAMLlena(pc_viejo) es un hecho observado (FC = 1.0).


In [12]:
print("¿Por qué NO se recomienda bajar los gráficos?\n")
for regla in porque_no("RecBajarGraficos", resultado.hechos, BASE, caso["equipo"]):
    print(regla["regla"], regla["logica"])
    for c in regla["condiciones"]:
        print(f"    {c['texto']:<28} -> {c['estado']}")

¿Por qué NO se recomienda bajar los gráficos?

R3 ∀x (CuelloBotellaGPU(x) ∧ GraficosAltos(x) → RecBajarGraficos(x))
    CuelloBotellaGPU(pc_viejo)   -> no_concluido
    GraficosAltos(pc_viejo)      -> desconocido


## 6. Casos de prueba y validación
Las reglas y conclusiones esperadas se escribieron **antes** de ejecutar el motor. Se cubren: casos típicos, varias causas a la vez, negación, combinación de FC, un caso **sin conclusión**, uno que activa **13 de 15 reglas** y dos **casos de borde** (dato desconocido y valores exactamente en el umbral).

In [13]:
filas = ["| Caso | Hechos iniciales | Reglas esperadas | Reglas obtenidas | Conclusiones obtenidas (FC) | Iter. | Estado |",
         "|---|---|---|---|---|---|---|"]
aciertos, usadas = 0, set()
for c in CASOS:
    h, _ = hechos_desde_formulario(c["datos"])
    r = ejecutar_inferencia(BASE, h, c["equipo"])
    obtenido = {p: v["certeza"] for p, v in r.hechos.items() if v["origen"] != "observado"}
    reglas = [p["regla"] for p in r.reglas_aplicadas]
    usadas |= set(reglas)
    ok = obtenido == c["esperado"] and reglas == c["reglas_esperadas"]
    aciertos += ok
    filas.append("| {} {} | {} | {} | {} | {} | {} | {} |".format(
        c["id"], c["nombre"], ", ".join(("" if v else "¬") + p for p, v in h.items()),
        ", ".join(c["reglas_esperadas"]) or "Ninguna", ", ".join(reglas) or "Ninguna",
        "<br>".join(f"{p} ({v:.2f})" for p, v in obtenido.items()) or "Ninguna",
        r.iteraciones, "✅" if ok else "❌"))
display(Markdown("\n".join(filas)))
print(f"Casos correctos: {aciertos}/{len(CASOS)} | reglas ejercitadas: {len(usadas)}/{len(REGLAS)}")

| Caso | Hechos iniciales | Reglas esperadas | Reglas obtenidas | Conclusiones obtenidas (FC) | Iter. | Estado |
|---|---|---|---|---|---|---|
| 1 PC con juego en ultra | FPSBajos, UsoGPUAlto, GraficosAltos | R1, R3 | R1, R3 | CuelloBotellaGPU (0.85)<br>RecBajarGraficos (0.77) | 3 | ✅ |
| 2 Portátil caliente | FPSBajos, TempAlta, RAMLlena, FPSCaenConTiempo, Tirones | R5, R7, R6, R8, R13 | R5, R7, R6, R8, R13 | ThermalThrottling (0.85)<br>FaltaRAM (0.80)<br>RecMantenimientoTermico (0.77)<br>RecAmpliarRAM (0.60)<br>RecCerrarProcesos (0.48) | 3 | ✅ |
| 3 Lag por WiFi | PingAlto, ConexionWiFi | R11, R12 | R11, R12 | ProblemaRedLocal (0.75)<br>RecCableEthernet (0.64) | 3 | ✅ |
| 4 Lag con cable (negación) | PingAlto, ¬ConexionWiFi | R14, R15 | R14, R15 | ProblemaServidor (0.60)<br>RecCambiarServidor (0.42) | 3 | ✅ |
| 5 PC viejo (combinación de FC) | FPSBajos, UsoCPUAlto, RAMLlena, JuegoEnHDD, Tirones | R2, R7, R9, R4, R8, R10, R13 | R2, R7, R9, R4, R8, R10, R13 | CuelloBotellaCPU (0.80)<br>FaltaRAM (0.80)<br>CuelloBotellaDisco (0.70)<br>RecCerrarProcesos (0.77)<br>RecAmpliarRAM (0.60)<br>RecMoverASSD (0.59) | 3 | ✅ |
| 6 Sin conclusión | ¬FPSBajos, ¬UsoGPUAlto, GraficosAltos | Ninguna | Ninguna | Ninguna | 1 | ✅ |
| 7 Todo falla (máxima cobertura) | FPSBajos, UsoGPUAlto, UsoCPUAlto, TempAlta, RAMLlena, PingAlto, GraficosAltos, JuegoEnHDD, ConexionWiFi, FPSCaenConTiempo, Tirones | R1, R2, R5, R7, R9, R11, R3, R4, R6, R8, R10, R12, R13 | R1, R2, R5, R7, R9, R11, R3, R4, R6, R8, R10, R12, R13 | CuelloBotellaGPU (0.85)<br>CuelloBotellaCPU (0.80)<br>ThermalThrottling (0.85)<br>FaltaRAM (0.80)<br>CuelloBotellaDisco (0.70)<br>ProblemaRedLocal (0.75)<br>RecBajarGraficos (0.77)<br>RecCerrarProcesos (0.77)<br>RecMantenimientoTermico (0.77)<br>RecAmpliarRAM (0.60)<br>RecMoverASSD (0.59)<br>RecCableEthernet (0.64) | 3 | ✅ |
| 8 Borde: conexión desconocida | PingAlto | Ninguna | Ninguna | Ninguna | 1 | ✅ |
| 9 Borde: valores en el umbral | ¬FPSBajos, UsoGPUAlto, ¬UsoCPUAlto, GraficosAltos | Ninguna | Ninguna | Ninguna | 1 | ✅ |

Casos correctos: 9/9 | reglas ejercitadas: 15/15


### 6.1 Casos especiales

In [14]:
# Sin conclusión: la respuesta lo indica explícitamente y el motor para en la iteración 1
r = diagnosticar(BASE, {"FPSBajos": False, "GraficosAltos": True}, "pc_ok")
print("sin_conclusion =", r["sin_conclusion"], "| iteraciones =", r["iteraciones"], "|", r["explicacion"])

# Dato desconocido vs. dato falso (negación)
for conexion in (None, False, True):
    h = {"PingAlto": True} if conexion is None else {"PingAlto": True, "ConexionWiFi": conexion}
    _, ra = encadenamiento_adelante(BASE, h)
    etiqueta = {None: "desconocida", False: "cable (¬WiFi)", True: "WiFi"}[conexion]
    print(f"Ping alto, conexión {etiqueta:<14} -> reglas: {[p['regla'] for p in ra] or 'ninguna'}")

sin_conclusion = True | iteraciones = 1 | No se aplicó ninguna regla: los hechos ingresados no alcanzan para un diagnóstico.
Ping alto, conexión desconocida    -> reglas: ninguna
Ping alto, conexión cable (¬WiFi)  -> reglas: ['R14', 'R15']
Ping alto, conexión WiFi           -> reglas: ['R11', 'R12']


## 7. Integración con la interfaz web
La aplicación Flask expone el motor como una API JSON. Aquí se prueba con el **cliente de pruebas de Flask** (no hace falta levantar el servidor). Para usar la interfaz:

```bash
pip install -r requirements.txt
python src/app.py          # abrir http://127.0.0.1:5000
```

In [15]:
from app import app
cliente = app.test_client()

print("GET /            ->", cliente.get("/").status_code)
base = cliente.get("/api/base").get_json()
print("GET /api/base    ->", len(base["reglas"]), "reglas,", len(base["casos"]), "casos precargados, ciclos:", base["ciclos"])

resp = cliente.post("/api/diagnosticar", json={"equipo": "pc_gamer", **CASOS[0]["datos"]}).get_json()
print("POST /api/diagnosticar ->", [(c["titulo"], c["certeza"]) for c in resp["diagnosticos"] + resp["recomendaciones"]])

resp = cliente.post("/api/diagnosticar", json={"uso_gpu": 150})
print("POST con GPU = 150 %  ->", resp.status_code, resp.get_json()["errores"])

resp = cliente.post("/api/porque", json={"datos": CASOS[2]["datos"], "conclusion": "ProblemaServidor"}).get_json()
print("POST /api/porque (¿por qué no ProblemaServidor si está por WiFi?) ->",
      [(c["texto"], c["estado"]) for c in resp["faltantes"][0]["condiciones"]])

GET /            -> 200
GET /api/base    -> 15 reglas, 9 casos precargados, ciclos: []
POST /api/diagnosticar -> [('La tarjeta gráfica (GPU) limita el rendimiento', 0.85), ('Bajar la calidad gráfica o activar DLSS/FSR', 0.77)]
POST con GPU = 150 %  -> 400 ['Uso de GPU: debe estar entre 0 y 100 %.']
POST /api/porque (¿por qué no ProblemaServidor si está por WiFi?) -> [('PingAlto(mi_pc)', 'cumple'), ('¬ConexionWiFi(mi_pc)', 'falso')]


Capturas de la interfaz (más detalle en `docs/manual_usuario.pdf`):

![Resultado del caso 5](docs/capturas/02_resultado.png)

![Árbol ¿Por qué?](docs/capturas/03_porque.png)

## 8. Análisis y conclusiones

### Análisis de resultados
- Los **9 casos** produjeron exactamente las reglas, en el orden y con los FC esperados; entre todos se usan las **15 reglas**.
- El encadenamiento se ve en niveles: iteración 1 → diagnósticos (R1, R2, R5, R7, R9, R11, R14); iteración 2 → recomendaciones (R3, R4, R6, R8, R10, R12, R13, R15); iteración 3 → nada nuevo, punto fijo.
- **Combinación de evidencia:** en el caso 5 la recomendación de cerrar procesos sube de 0.56 (solo R4) a 0.77 al sumar R13; por eso aparece primera en la interfaz.
- **Negación con tres estados:** con la conexión desconocida (caso 8) el sistema no culpa ni al WiFi ni al servidor, en vez de asumir "no es WiFi" por omisión.
- **Umbrales:** en el caso 9, 30 FPS no cumplen `fps < 30`, así que R1 no se dispara aunque la GPU esté al 95 %.

### Limitaciones
- Los FC salen de nuestra experiencia como jugadores, no de datos medidos.
- La combinación MYCIN solo ocurre en una conclusión hoja (RecCerrarProcesos); si se combinara un diagnóstico intermedio, los FC de las recomendaciones que ya dependían de él no se recalcularían.
- Umbrales fijos: 89 % de CPU no es "alto" y 90 % sí; no hay lógica difusa.
- Un solo equipo por consulta y sin memoria entre consultas.

### Posibles mejoras
- Lógica difusa para los umbrales y FC calibrados con mediciones reales.
- Reglas de drivers, fuente de poder y resolución; modo "entrevista" que pregunte solo los datos que faltan (encadenamiento hacia atrás usando `porque_no`).
- Importar el log de MSI Afterburner directamente.

### Reflexión
Pasar de notebook a aplicación nos obligó a separar responsabilidades: la base de conocimiento es solo datos, el motor no sabe nada de la web y la web no sabe nada de reglas. Eso permitió probar el motor con `unittest` y en este notebook, y agregar R13–R15 sin tocar la interfaz. La parte más interesante fue la negación: decidir que "no sé" no es lo mismo que "no" cambió el diseño de los hechos de dos a tres estados.